# 03 · Collect: check what the crawler saved

The crawl itself runs from the terminal, because it takes hours and has to survive stops:

```bash
cd backend
python -m discovery collect --step list            # search requests only, ~15 min
python -m discovery collect --step sample          # no requests: each slice's quota, seed 42
python -m discovery collect --step fetch --limit 20   # try 20 first, read them below
python -m discovery collect                        # the rest; Ctrl+C any time, rerun to resume
```

This notebook only **reads** `data/raw/` and runs the Phase 3 checks. Run it after each step.

In [1]:
%load_ext autoreload
%autoreload 2

import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from discovery.collect.crawler import iter_jsonl, load_plan
from discovery.config import BACKEND, PATHS, SEED, load_config, set_seed

set_seed()
cfg = load_config()
RAW = PATHS["raw"]
plan = load_plan(BACKEND / "config" / "slices.csv")
print(f"{len(plan)} search pieces | planned take {plan['take'].sum():,} | raw folder {RAW.relative_to(BACKEND)}")

452 search pieces | planned take 11,421 | raw folder data/raw


## 1. List: did every piece come back whole?

In [2]:
listing = pd.DataFrame([{"piece_id": r["piece_id"], "full_name": r["item"]["full_name"]} for r in iter_jsonl(RAW / "listing.jsonl")])
got = listing.groupby("piece_id").size().rename("listed")
cmp = plan.set_index("piece_id")[["domain", "framework", "count"]].join(got).fillna({"listed": 0})
cmp["expected"] = np.minimum(cmp["count"], 1000)
cmp["diff"] = cmp["listed"] - cmp["expected"]
print(f"pieces done: {(cmp['listed'] > 0).sum()} of {(cmp['count'] > 0).sum()} | items listed: {len(listing):,} | unique repositories: {listing['full_name'].nunique():,}")
print(f"listed vs counted in Phase 2: {int(cmp['listed'].sum()):,} vs {int(cmp['expected'].sum()):,} (counts drift a little day to day)")
cmp.sort_values("diff").head(8)

pieces done: 452 of 452 | items listed: 212,348 | unique repositories: 128,814
listed vs counted in Phase 2: 212,348 vs 212,569 (counts drift a little day to day)


,domain,framework,count,listed,expected,diff
piece_id,,,,,,
p111,blog,spring-boot,731,499,731,-232
p264,todo,react,991,988,991,-3
p204,todo,express,765,763,765,-2
p276,todo,react,946,944,946,-2
p138,blog,react,734,732,734,-2
p420,portfolio,react,673,672,673,-1
p136,blog,react,970,969,970,-1
p406,portfolio,react,572,571,572,-1


**Check:** each piece lists about what Phase 2 counted. A few more or fewer is normal (stars change daily). A piece with 0 listed but a count above 0 did not finish: rerun `--step list`.

## 2. Duplicates: how many slices found the same repository?

In [3]:
per_repo = listing.groupby("full_name")["piece_id"].nunique()
print(f"{(per_repo > 1).mean():.1%} of repositories were found by more than one piece; most found: {per_repo.idxmax()} ({per_repo.max()} pieces)")
per_repo.value_counts().sort_index().head(8).rename("repositories").rename_axis("found by n pieces").to_frame().T

29.1% of repositories were found by more than one piece; most found: turhobr/selfhost.directory (77 pieces)


found by n pieces,1,2,3,4,5,6,7,8
repositories,91370,25991,3587,3616,387,1860,96,420


## 3. Sample: each slice's quota, at random

In [4]:
sel = pd.read_csv(RAW / "selected.csv", keep_default_na=False)
sel["domains"] = sel["selected_by"].str.split(";").map(lambda s: sorted({x.split("|")[0] for x in s}))
print(f"selected {len(sel):,} repositories ({sel['is_other'].sum():,} other = {sel['is_other'].mean():.0%}) | planned {plan['take'].sum():,} before duplicates")
sel.explode("domains")["domains"].value_counts().rename("selected").to_frame().T

selected 8,657 repositories (986 other = 11%) | planned 11,421 before duplicates


domains,blog,other,todo,portfolio,e-commerce,crm,chat application,inventory management,booking system,library management,point of sale,school management,hospital management
selected,997,996,993,946,944,837,758,728,577,575,452,337,298


## 4. Fetch: the Phase 3 checks

In [5]:
repos = list(iter_jsonl(RAW / "repos.jsonl"))
names = [r["full_name"] for r in repos]
errors = list(iter_jsonl(RAW / "errors.jsonl")) if (RAW / "errors.jsonl").exists() else []
print(f"lines in repos.jsonl: {len(names):,} | unique full_name: {len(set(names)):,} | equal: {len(names) == len(set(names))}")
print(f"fetched {len(names):,} of {len(sel):,} selected ({len(names) / len(sel):.0%}) | errors {len(errors)}")
print(f"no README: {sum(r['readme'] == '' for r in repos):,} (stored with readme = '', not skipped)")
print(f"no languages: {sum(not r['languages'] for r in repos):,} | no contributors: {sum(not r['contributors'] for r in repos):,}")
pd.DataFrame(errors).groupby("status").size() if errors else "no errors" 

lines in repos.jsonl: 8,657 | unique full_name: 8,657 | equal: True
fetched 8,657 of 8,657 selected (100%) | errors 0
no README: 127 (stored with readme = '', not skipped)
no languages: 1,391 | no contributors: 228


'no errors'

In [6]:
ex = repos[0]
{"full_name": ex["full_name"], "stars": ex["search"].get("stargazers_count"), "found_by": ex["found_by"], "selected_by": ex["selected_by"],
 "readme (start)": ex["readme"][:300], "languages": ex["languages"], "contributors (first 3)": ex["contributors"][:3]}

{'full_name': '00xZEROx00/awesome-list',
 'stars': 6,
 'found_by': ['p001', 'p069'],
 'selected_by': ['inventory management|express'],
 'readme (start)': '# Awesome List 🌟 [![Awesome](https://awesome.re/badge.svg)](https://awesome.re) ![GitHub Actions Workflow Status](https://img.shields.io/github/actions/workflow/status/00xBAD/awesome-list/.github%2Fworkflows%2Fbuild.yml)\n\nA curated list of awesome tools, frameworks, utilities, and resources spanning',
 'languages': {'Shell': 2505},
 'contributors (first 3)': [{'login': '00xZEROx00',
   'type': 'User',
   'contributions': 4},
  {'login': 'github-actions[bot]', 'type': 'Bot', 'contributions': 2}]}

## What we found

*(fill in after the crawl: pieces listed vs counted, share of duplicates, repositories without README, errors, how long it took)*